<h1><font color="#113D68" size=6>Deep Learning en Python con PyTorch</font></h1>

<h1><font color="#113D68" size=5>Parte 4. Mejorar la Generalización en modelos</font></h1>

<h1><font color="#113D68" size=4>9. Gestionar el proceso de entrenamiento
</font></h1>

<br><br>
<div style="text-align: right">
<font color="#113D68" size=3>Manuel Castillo Cara</font><br>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web](https://www.manuelcastillo.eu/udemy/)


---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>

* [0. Contexto](#section0)
* [1. Guardar Modelos de Redes Neuronales](#section1)
* [2. Guardar con detención anticipada](#section2)

---
<a id="section0"></a>
# <font color="#004D7F" size=6> 0. Contexto</font>

Un modelo de aprendizaje profundo grande puede tardar mucho tiempo en entrenarse. Pierdes mucho trabajo si el proceso de entrenamiento se interrumpe en medio. Pero a veces, realmente deseas interrumpir el proceso de entrenamiento porque sabes que continuar no te dará un mejor modelo. En este capítulo, descubrirás cómo controlar el bucle de entrenamiento en PyTorch para que puedas reanudar un proceso interrumpido o detener el bucle de entrenamiento anticipadamente.

Después de completar esta clase, sabrás:
- La importancia de guardar modelos de redes neuronales durante el entrenamiento.
- Cómo guardar un modelo durante el entrenamiento y restaurarlo más tarde.
- Cómo terminar el bucle de entrenamiento anticipadamente con el guardado de modelos.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
[PyTorch tutorials.](https://pytorch.org/tutorials/)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch Documentation: [`torch.utils.data`](https://pytorch.org/docs/stable/data.html)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch Documentation: [`torch.utils.checkpoint`](https://pytorch.org/docs/stable/checkpoint.html)

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section1"></a>
# <font color="#004D7F" size=6>1. Guardar Modelos de Redes Neuronales</font>

Esta implementación permite una gestión eficiente del estado del modelo durante el entrenamiento, facilitando la recuperación en caso de interrupciones y la continuación del entrenamiento desde puntos específicos.

- Concepto de estados en sistemas:
  - Posibilidad de guardar y restaurar estados

  - Permite retroceder a puntos específicos en el tiempo

- Analogía con Microsoft Word:
  - Guardar múltiples versiones de un documento

  - Facilita la reversión de ediciones

- Aplicación en procesos de larga duración:
  - Técnica de tolerancia a fallos

  - Toma instantáneas del estado del sistema

- Ventajas del guardado de aplicaciones:
  - Permite reanudar desde una instantánea en caso de falla

  - Útil como punto de partida para nuevas ejecuciones

- Implementación de lógica de guardado en PyTorch:
  - Crear función personalizada para guardar y reanudar

  - Proceso:

    1. Guardar los pesos del modelo
    
    2. Cargar los pesos cuando sea necesario


In [ ]:
import torch

def checkpoint(model, filename):
    torch.save(model.state_dict(), filename)

def resume(model, filename):
    model.load_state_dict(torch.load(filename))


A continuación se muestra cómo generalmente entrenarías un modelo de PyTorch. El conjunto de datos utilizado se obtiene de la plataforma OpenML. Es un conjunto de datos de clasificación binaria. En este ejemplo, se utiliza PyTorch DataLoader para hacer el bucle de entrenamiento más conciso.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [_Fetch dataset from OpenML_](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split, default_collate
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder

data = fetch_openml("electricity", version=1, parser="auto")  # se descarga de OpenML


X = data['data'].astype('float').values
y = data['target']
encoder = LabelEncoder()
encoder.fit(y)
y = encoder.transform(y)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)


trainset, testset = random_split(TensorDataset(X, y), [0.7, 0.3])

model = nn.Sequential(
    nn.Linear(8, 12),
    nn.ReLU(),
    nn.Linear(12, 12),
    nn.ReLU(),
    nn.Linear(12, 1),
    nn.Sigmoid(),
)

n_epochs = 100
loader = DataLoader(trainset, shuffle=True, batch_size=32)
X_test, y_test = default_collate(testset)

loss_fn = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in loader:
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    model.eval()
    y_pred = model(X_test)
    acc = (y_pred.round() == y_test).float().mean()
    print(f"Final de la época {epoch}: accuracy = {float(acc)*100:.2f}%")

Esta implementación proporciona una robusta estrategia de tolerancia a fallos para el entrenamiento de modelos, permitiendo una fácil reanudación y mantenimiento de la consistencia en experimentos a largo plazo.

- Adición de puntos de control al bucle de entrenamiento:
  - Implementar al final del bucle externo

  - Coincide con la validación del modelo en el conjunto de prueba

- Técnica de tolerancia a fallos:
  - Permite reanudar desde una época específica

  - Requiere pocas líneas de código adicionales antes del bucle de entrenamiento

- Ejemplo de reanudación:
  - Escenario: Interrupción en época 8, último punto de control en época 7

  - Solución: Establecer `start_epoch = 8`

- Consideración importante:
  - La función `random_split()` puede generar divisiones diferentes en cada ejecución

  - Puede afectar la consistencia entre ejecuciones interrumpidas y reanudadas

- Solución para mantener consistencia:
  - Implementar una forma consistente de crear conjuntos de datos

  - Opción: Guardar los datos divididos para reutilizarlos


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split, default_collate
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder

model = nn.Sequential(
    nn.Linear(8, 12),
    nn.ReLU(),
    nn.Linear(12, 12),
    nn.ReLU(),
    nn.Linear(12, 1),
    nn.Sigmoid(),
)

n_epochs = 100
start_epoch = 0
loader = DataLoader(trainset, shuffle=True, batch_size=32)
X_test, y_test = default_collate(testset)
loss_fn = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

def checkpoint(model, filename):
    torch.save(model.state_dict(), filename)

def resume(model, filename):
    model.load_state_dict(torch.load(filename))

# reanudar desde el último checkpoint si start_epoch > 0
if start_epoch > 0:
    resume_epoch = start_epoch - 1
    resume(model, f"epoch-{resume_epoch}.pth")

for epoch in range(start_epoch, n_epochs):
    model.train()
    for X_batch, y_batch in loader:
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    model.eval()
    y_pred = model(X_test)
    acc = (y_pred.round() == y_test).float().mean()
    print(f"Final de la época {epoch}: accuracy = {float(acc)*100:.2f}%")
    checkpoint(model, f"epoch-{epoch}.pth")  # guardar al final de cada época

- Verás un número de archivos creados en tu directorio de trabajo. 

- Este código guardará el modelo del epóca 7, por ejemplo, en el archivo `epoch-7.pth`. 

- Cada uno de estos archivos es un archivo ZIP con el modelo guardado en formato pickle. 

- No hay nada que impida guardar dentro del bucle interno, pero debido al overhead que incurre, no es una buena idea guardar demasiado frecuente.


- A veces, hay estados fuera del modelo que también puedes querer guardar. 

- Un ejemplo particular es el optimizador, en el que, en casos como Adam, hay momentos que se ajustan dinámicamente. 

- Si reinicias tu bucle de entrenamiento, es posible que también desees restaurar el momento en el optimizador. 

- No es difícil de hacer. La idea es hacer tu función `checkpoint()` más complicada.

In [ ]:
def checkpoint(model, optimizer, filename):
    torch.save({
        'optimizer': optimizer.state_dict(),
        'model': model.state_dict(),
    }, filename)

y, en consecuencia, cambiar tu función `resume()`:

In [ ]:
def resume(model, optimizer, filename):
    checkpoint = torch.load(filename)
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])

Esto funciona porque en PyTorch, las funciones `torch.save()` y `torch.load()` están respaldadas por pickle, así que puedes utilizarlas con una lista o un contenedor de diccionario.

- Para juntar todo, a continuación se muestra el código completo. 

- Fijarse que en la función `checkpoint()` hemos añadido el optimizador.

In [ ]:
n_epochs = 100
start_epoch = 0
loader = DataLoader(trainset, shuffle=True, batch_size=32)
X_test, y_test = default_collate(testset)
loss_fn = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

if start_epoch > 0:
    resume_epoch = start_epoch - 1
    resume(model, optimizer, f"epoch-{resume_epoch}.pth")

for epoch in range(start_epoch, n_epochs):
    model.train()
    for X_batch, y_batch in loader:
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    model.eval()
    y_pred = model(X_test)
    acc = (y_pred.round() == y_test).float().mean()
    print(f"Final de la época {epoch}: accuracy = {float(acc)*100:.2f}%")
    checkpoint(model, optimizer, f"epoch-{epoch}.pth")

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section2"></a>
# <font color="#004D7F" size=6>2. Guardar con detención anticipada</font>

- El guardado no solo se utiliza para la tolerancia a fallos. 

- También puedes usarlo para mantener tu mejor modelo. 

- Cómo definir cuál es el mejor es subjetivo, pero considerar la puntuación del conjunto de test. 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split, default_collate
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder

data = fetch_openml("electricity", version=1, parser="auto")

X = data['data'].astype('float').values
y = data['target']
encoder = LabelEncoder()
encoder.fit(y)
y = encoder.transform(y)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

trainset, testset = random_split(TensorDataset(X, y), [0.7, 0.3])

def checkpoint(model, filename):
    torch.save(model.state_dict(), filename)

def resume(model, filename):
    model.load_state_dict(torch.load(filename))

model = nn.Sequential(
    nn.Linear(8, 12),
    nn.ReLU(),
    nn.Linear(12, 12),
    nn.ReLU(),
    nn.Linear(12, 1),
    nn.Sigmoid(),
)

n_epochs = 10000  # cota alta: el early stopping decide cuándo parar  
loader = DataLoader(trainset, shuffle=True, batch_size=32)
X_test, y_test = default_collate(testset)
loss_fn = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

Este enfoque proporciona una estrategia robusta para el entrenamiento y guardado de modelos, asegurando que se conserve la versión más efectiva del modelo mientras se optimiza el proceso de entrenamiento.

- Modificación del bucle `for`:
  - Introducción de `best_accuracy` para seguimiento

  - Rango de accuracy: 0 a 100 (porcentual)

- Guardado del mejor modelo:
  - Archivo: `best_model.pth`

  - Se actualiza cuando se observa un accuracy más alto

- Restauración del mejor modelo:
  - Se realiza después del bucle de entrenamiento

  - Uso de la función `resume()` previamente creada

- Optimización del entrenamiento:
  - Implementación de detención anticipada

  - Ahorra tiempo si las mejoras son poco probables

  - Asegura la retención del modelo con mejor rendimiento

  - Permite flexibilidad en estrategias de guardado

  - Implementa optimizaciones como la detención anticipada

In [ ]:
early_stop_thresh = 5  # épocas sin mejora antes de parar
best_accuracy = -1
best_epoch = -1

for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in loader:
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    model.eval()
    y_pred = model(X_test)
    acc = (y_pred.round() == y_test).float().mean()
    acc = float(acc) * 100
    print(f"Final de la época {epoch}: accuracy = {acc:.2f}%")

    if acc > best_accuracy:
        best_accuracy = acc
        best_epoch =epoch
        checkpoint(model, "best_model.pth")
    elif epoch - best_epoch > early_stop_thresh:
        print("Finaliza el entrenamiento en la época %d" % epoch)
        break  

resume(model, "best_model.pth")  # recuperar el mejor modelo

- Estrategia simple de detención anticipada:
  - Establecer un umbral `k` de épocas
  
  - Detener si no hay mejora durante las últimas `k` épocas

- Implementación:
  - Variable `early_stop_thresh` (ejemplo: 5)

  - `best_epoch` para recordar la época del mejor modelo

- Ventajas:
  - Alivia la necesidad de ajustar precisamente `n_epochs`

  - `n_epochs` puede ser un número máximo mayor al necesario

- Prevención de sobreajuste:
  - Detiene el entrenamiento si el rendimiento empeora

  - Restaura el mejor punto de guardado

- Resultados típicos:
  - Detención antes del máximo de épocas establecido

  - Ejemplo: Detención en época 17, mejor modelo de época 11


- Consideraciones adicionales:
  - Ajustar `early_stop_thresh` según la naturaleza del problema

  - Balancear entre tiempo de entrenamiento y potencial de mejora

- Beneficios:
  - Optimiza tiempo de entrenamiento

  - Reduce riesgo de sobreajuste

  - Mantiene el modelo con mejor rendimiento

<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>